# Analyse de l'influence des modèles LLM sur le choix de mobilité

Objectif : resoumettre à chaque provider les prompts de sélection d'itinéraire issus de `llm_exchanges.jsonl`  
(système débutant par *"Tu es un expert en mobilité urbaine à Toulouse, France."*),  
puis comparer les choix de mode de transport produits par chaque modèle.

In [ ]:
import sys
import json
import time
import threading
import numpy as np
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

import yaml
import pandas as pd
from tqdm.notebook import tqdm

# ── Chemins ────────────────────────────────────────────────────────────────
PROJECT_ROOT = Path("../..")
sys.path.insert(0, str(PROJECT_ROOT))

EXCHANGES_FILE = PROJECT_ROOT / "experiments" / "current" / "llm_exchanges.jsonl"
SCHEMAS_FILE   = PROJECT_ROOT / "llm_module" / "prompts" / "schemas.json"

SYSTEM_PROMPT_PREFIX = "Tu es un expert en mobilité urbaine à Toulouse, France."
CATEGORY             = "itinary_multi_agent"
SAMPLE_SIZE = 100

# ── Sélection du prompt système ─────────────────────────────────────────────
PROMPT_NAME = "persona_v3"

with open("prompts.yaml") as f:
    _prompts_db = yaml.safe_load(f)

_selected            = _prompts_db["prompts"][PROMPT_NAME]
ACTIVE_SYSTEM_PROMPT = _selected.get("content")  # None = conserver le prompt d'origine du JSONL
_variant             = PROMPT_NAME
OUTPUT_CSV           = Path("results") / f"influence_results_{_variant}.csv"
print(f"Prompt  : {PROMPT_NAME!r}")
print(f"Variant : {_variant}  →  {OUTPUT_CSV}")

## 1 — Chargement du JSONL et filtrage des entrées éligibles

In [ ]:
import random

def parse_multiline_jsonl(path: Path) -> list[dict]:
    """Parse un fichier JSONL dont les objets JSON sont multi-lignes."""
    content = path.read_text(encoding="utf-8").strip()
    decoder = json.JSONDecoder()
    entries, pos = [], 0
    while pos < len(content):
        # Sauter les espaces/retours à la ligne entre objets
        stripped = content[pos:].lstrip()
        pos += len(content[pos:]) - len(stripped)
        if not stripped:
            break
        obj, offset = decoder.raw_decode(stripped)
        entries.append(obj)
        pos += offset
    return entries


all_entries = parse_multiline_jsonl(EXCHANGES_FILE)

eligible = [
    e for e in all_entries
    if e.get("messages", [{}])[0].get("content", "").startswith(SYSTEM_PROMPT_PREFIX)
]

if len(eligible) > SAMPLE_SIZE:
    random.seed(42)
    eligible = random.sample(eligible, SAMPLE_SIZE)

print(f"Entrées totales  : {len(all_entries)}")
print(f"Entrées éligibles: {len(eligible)} (max {SAMPLE_SIZE})")

## 2 — Chargement du schéma de réponse et des providers actifs

In [ ]:
from llm_module.adapters.base import get_adapter
from llm_module.settings.models import InternalMessage, InternalRequest
from llm_module.tasks.llm_config import settings

with open(SCHEMAS_FILE) as f:
    RESPONSE_SCHEMA = json.load(f)[CATEGORY]

providers = settings.providers  # Dict[str, ProviderConfig] — seulement ceux avec clé API

print(f"Providers actifs ({len(providers)}) :")
for name, cfg in providers.items():
    print(f"  {name:<30} model={cfg.default_model}  rpm={cfg.rpm_limit}")

## 3 — Fonction d'appel unitaire par provider

In [ ]:
from llm_module.adapters.base import ProviderClientError, ProviderServerError, ProviderParseError
from llm_module.worker.task_worker import _parse_ratelimit_reset_seconds

MAX_RETRIES    = 5
MAX_RETRY_WAIT = 300.0  # secondes — au-delà, quota journalier épuisé → abandon


def call_provider(provider_name: str, entry: dict) -> list[dict]:
    """Appel brut — construit InternalRequest et retourne les lignes agent."""
    raw_messages = entry["messages"]
    if ACTIVE_SYSTEM_PROMPT is not None:
        raw_messages = [
            {**raw_messages[0], "content": ACTIVE_SYSTEM_PROMPT},
            *raw_messages[1:],
        ]
    system_prompt = raw_messages[0]["content"]
    messages = [InternalMessage(role=m["role"], content=m["content"]) for m in raw_messages]
    request = InternalRequest(
        provider=provider_name,
        messages=messages,
        response_schema=RESPONSE_SCHEMA,
    )
    llm_output, _, _ = get_adapter(provider_name).call(request)
    user_prompt = entry["messages"][1]["content"]
    return [
        {
            "user_prompt":   user_prompt,
            "system_prompt": system_prompt,
            "provider":      provider_name,
            "agent_id":      agent.agent_id,
            "chosen_index":  agent.chosen_index,
            "mode":          agent.mode,
            "reason":        agent.reason,
        }
        for agent in llm_output.agents
    ]


def call_provider_with_retry(provider_name: str, entry: dict) -> list[dict]:
    """
    Appel avec retry :
    - 429 : attend le délai indiqué par x-ratelimit-reset (max MAX_RETRY_WAIT, sinon abandon)
    - 5xx : backoff exponentiel (1, 2, 4, 8, 16 s)
    - ParseError : 1 retry immédiat (réponse tronquée possible)
    """
    parse_attempts = 0
    for attempt in range(MAX_RETRIES + 1):
        try:
            return call_provider(provider_name, entry)

        except ProviderClientError as exc:
            if exc.status_code == 429 and attempt < MAX_RETRIES:
                wait = _parse_ratelimit_reset_seconds(exc.ratelimit_reset)
                if wait > MAX_RETRY_WAIT:
                    raise  # quota journalier épuisé
                time.sleep(wait)
            else:
                raise

        except ProviderServerError:
            if attempt < MAX_RETRIES:
                time.sleep(2 ** attempt)
            else:
                raise

        except ProviderParseError:
            if parse_attempts < 1:
                parse_attempts += 1
                time.sleep(2.0)
            else:
                raise

## 4 — Exécution parallèle (un thread par provider, rate limiting intégré)

In [ ]:
import shutil
from datetime import datetime

# ── Détection du mode reprise ──────────────────────────────────────────────
def _get_experiment_slug() -> str | None:
    """Slug YYYY-MM-DD_HH_MM depuis le champ 'time' du 1er échange."""
    try:
        first = parse_multiline_jsonl(EXCHANGES_FILE)[0]
        dt = datetime.fromisoformat(first["time"])
        return dt.strftime("%Y-%m-%d_%H_%M")
    except Exception:
        return None

_slug = _get_experiment_slug()
_experiment_archive = (PROJECT_ROOT / "experiments" / "archive" / _slug) if _slug else None
RESUME_MODE = bool(_experiment_archive and _experiment_archive.exists())

if RESUME_MODE:
    print(f"Mode reprise — archive détectée : {_experiment_archive}")
    print(f"Combinaisons (prompt, provider) déjà dans {OUTPUT_CSV} seront ignorées.")
else:
    _results_dir = Path("results")
    if _results_dir.exists() and any(_results_dir.iterdir()):
        _ts = datetime.now().strftime("%Y-%m-%d_%H%M%S")
        _archive = Path("results_archive") / _ts
        _archive.mkdir(parents=True, exist_ok=True)
        for _f in _results_dir.iterdir():
            shutil.copy2(_f, _archive / _f.name)
        print(f"Résultats précédents archivés → {_archive}")
    else:
        print("Aucun résultat précédent à archiver.")


In [ ]:
_csv_lock      = threading.Lock()
_prompt_to_idx: dict[str, int] = {}
_prompt_counter = 0

MAX_CONSECUTIVE_FAILURES = 10


def _get_prompt_idx(user_prompt: str) -> int:
    """Retourne l index stable du prompt (créé à la première rencontre, thread-safe)."""
    global _prompt_counter
    if user_prompt not in _prompt_to_idx:
        _prompt_to_idx[user_prompt] = _prompt_counter
        _prompt_counter += 1
    return _prompt_to_idx[user_prompt]


def process_provider(provider_name: str, entries: list[dict], rpm_limit: int) -> list[dict]:
    delay = 60.0 / rpm_limit
    rows = []
    consecutive_failures = 0
    bar = tqdm(entries, desc=provider_name, leave=True, position=list(providers).index(provider_name))

    for entry in bar:
        user_prompt = entry["messages"][1]["content"]

        if RESUME_MODE and (user_prompt, provider_name) in _done_pairs:
            continue

        try:
            new_rows = call_provider_with_retry(provider_name, entry)
            consecutive_failures = 0
            with _csv_lock:
                prompt_idx = _get_prompt_idx(new_rows[0]["user_prompt"])
                for r in new_rows:
                    r["prompt_idx"] = prompt_idx
                rows.extend(new_rows)
                write_header = not OUTPUT_CSV.exists()
                pd.DataFrame(new_rows).to_csv(OUTPUT_CSV, mode="a", header=write_header, index=False)
        except Exception as exc:
            consecutive_failures += 1
            bar.write(f"[{provider_name}] ABANDON ({consecutive_failures}/{MAX_CONSECUTIVE_FAILURES}): {exc}")
            if consecutive_failures >= MAX_CONSECUTIVE_FAILURES:
                bar.write(f"[{provider_name}] {MAX_CONSECUTIVE_FAILURES} échecs consécutifs — provider abandonné")
                break

        time.sleep(delay)

    return rows


OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)

if RESUME_MODE and OUTPUT_CSV.exists():
    _existing_df = pd.read_csv(OUTPUT_CSV)
    _done_pairs: set[tuple[str, str]] = set(
        zip(_existing_df["user_prompt"], _existing_df["provider"])
    )
    for _up in _existing_df["user_prompt"].unique():
        _get_prompt_idx(_up)
    print(f"Reprise : {len(_done_pairs)} paires (prompt, provider) déjà calculées")
else:
    _done_pairs = set()
    if OUTPUT_CSV.exists():
        OUTPUT_CSV.unlink()
    _prompt_to_idx.clear()
    _prompt_counter = 0

all_rows: list[dict] = []

with ThreadPoolExecutor(max_workers=len(providers)) as executor:
    futures = {
        executor.submit(process_provider, name, eligible, cfg.rpm_limit): name
        for name, cfg in providers.items()
    }
    for future in as_completed(futures):
        provider_name = futures[future]
        try:
            all_rows.extend(future.result())
        except Exception as exc:
            print(f"[{provider_name}] thread échoué: {exc}")

print(f"Terminé — {len(all_rows)} nouvelles lignes collectées")


## 5 — Aperçu des résultats

In [ ]:
df = pd.read_csv(OUTPUT_CSV)
print(f"Lignes totales : {len(df)}")
print(f"Providers      : {df['provider'].nunique()}")
print(f"Prompts uniques: {df['user_prompt'].nunique()}")
df.head(5)

## 6 — Analyse des distributions (`influence_results.csv`)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
from scipy.stats import entropy as sp_entropy
from pathlib import Path

# ── Chargement ───────────────────────────────────────────────────────────────
OUTPUT_CSV = Path("results") / "influence_results_persona_v2.csv"
df_raw = pd.read_csv(OUTPUT_CSV)

# Exclure les lignes en échec (chosen_index hors 1-5)
df_valid = df_raw[df_raw["chosen_index"].between(1, 5)].copy()
excluded_rows = len(df_raw) - len(df_valid)
print(f"Lignes exclues (échecs) : {excluded_rows}")

# Exclure les providers avec trop peu de réponses valides (runs quasi-échoués)
MIN_RESPONSES = 20
counts = df_valid["provider"].value_counts()
excluded_providers = counts[counts < MIN_RESPONSES].index.tolist()
if excluded_providers:
    print(f"Providers exclus (< {MIN_RESPONSES} réponses valides) : {excluded_providers}")
df = df_valid[~df_valid["provider"].isin(excluded_providers)].copy()

# ── Palette officielle modes de transport ────────────────────────────────────
MODE_COLORS = {
    "Voiture": "red",
    "Vélo":    "purple",
    "TC":      "green",
    "Marche":  "cyan",
    "Autre":   "gray",
}

def categorize_mode(m: str) -> str:
    if not isinstance(m, str):
        return "Autre"
    ml = m.lower()
    if any(k in ml for k in ("voiture", "car", "conducteur")):
        return "Voiture"
    if any(k in ml for k in ("vélo", "velo", "bicycle", "cycling", "vélib")):
        return "Vélo"
    if any(k in ml for k in ("bus", "metro", "métro", "tram", "transit", "transports en commun","public_transport")):
        return "TC"
    if any(k in ml for k in ("marche", "foot", "walk")):
        return "Marche"
    print(f"Mode non classifié (Autre) : '{m}'")
    return "Autre"

df["mode_cat"] = df["mode"].apply(categorize_mode)

# Noms courts pour les graphiques
providers_order = sorted(df["provider"].unique())
SHORT = {p: p.replace("google_", "g-").replace("groq_", "q-").replace("cerebras_", "cb-") for p in providers_order}

print(f"\nDonnées analysées : {len(df)} lignes, {len(providers_order)} modèles, {df['user_prompt'].nunique()} prompts uniques")
print("\nRéponses valides par modèle :")
print(df["provider"].value_counts().rename(index=SHORT).to_string())

# Vérifier les modes non classifiés
autre = df[df["mode_cat"] == "Autre"]["mode"].value_counts()
if len(autre):
    print("\nModes non classifiés (Autre) :")
    print(autre.head(10))


### Graphique 1 — Diagramme à barres empilées 100% : fréquence relative des choix de mode

In [ ]:
cats = list(MODE_COLORS.keys())
pivot = (
    df.groupby(["provider", "mode_cat"])
      .size()
      .unstack(fill_value=0)
      .reindex(columns=cats, fill_value=0)
      .loc[providers_order]
)
pct = pivot.div(pivot.sum(axis=1), axis=0) * 100

fig, ax = plt.subplots(figsize=(12, 5))
bottom = np.zeros(len(pct))
for cat in cats:
    vals = pct[cat].values
    bars = ax.bar(range(len(pct)), vals, bottom=bottom,
                  color=MODE_COLORS[cat], label=cat, width=0.65, edgecolor="white", linewidth=0.4)
    for i, (v, b) in enumerate(zip(vals, bottom)):
        if v > 3:
            ax.text(i, b + v / 2, f"{v:.0f}%", ha="center", va="center",
                    fontsize=7.5, color="white", fontweight="bold")
    bottom += vals

ax.set_xticks(range(len(pct)))
ax.set_xticklabels([SHORT[p] for p in providers_order], rotation=30, ha="right", fontsize=9)
ax.set_ylabel("Pourcentage (%)")
ax.set_ylim(0, 100)
ax.set_title(f"Fréquence relative des choix de mode par modèle ({PROMPT_NAME})", fontsize=13, fontweight="bold")
ax.legend(loc="lower left", bbox_to_anchor=(1.02, 0), framealpha=0.85, fontsize=9)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.savefig("results/g1_stacked_bar.png", dpi=150)
plt.show()


### Graphique 2 — Entropie de Shannon : diversité des choix par modèle

In [ ]:
# Entropie calculée sur chosen_index (1-5), base 2, normalisée sur log2(5)
MAX_ENTROPY = np.log2(5)

def shannon_entropy(series):
    counts = series.value_counts(normalize=True)
    return sp_entropy(counts, base=2)

entropies = (
    df.groupby("provider")["chosen_index"]
      .apply(shannon_entropy)
      .reindex(providers_order)
)

fig, ax = plt.subplots(figsize=(10, 4))
colors_bar = ["#2196F3" if e >= MAX_ENTROPY * 0.75 else "#90CAF9" for e in entropies]
bars = ax.bar(range(len(entropies)), entropies, color=colors_bar, edgecolor="white", linewidth=0.5)

ax.axhline(MAX_ENTROPY, color="orange", linestyle="--", linewidth=1.2, label=f"Entropie max ({MAX_ENTROPY:.2f} bits)")
ax.set_xticks(range(len(entropies)))
ax.set_xticklabels([SHORT[p] for p in providers_order], rotation=30, ha="right", fontsize=9)
ax.set_ylabel("Entropie de Shannon (bits)")
ax.set_title("Entropie de Shannon des choix de mode par modèle", fontsize=13, fontweight="bold")
ax.legend(fontsize=9)
ax.set_ylim(0, MAX_ENTROPY * 1.15)

for i, v in enumerate(entropies):
    ax.text(i, v + 0.02, f"{v:.2f}", ha="center", va="bottom", fontsize=8)

ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.savefig("results/g2_entropy.png", dpi=150)
plt.show()


### Graphique 3 — Carte de chaleur d'accord pair-à-pair entre modèles

In [ ]:
# Pour chaque prompt : mode majoritaire par (prompt, provider)
pivot_agree = (
    df.groupby(["user_prompt", "provider"])["chosen_index"]
      .agg(lambda x: int(x.mode().iloc[0]))
      .unstack("provider")
      .reindex(columns=providers_order)
)

n = len(providers_order)
agree_matrix = np.full((n, n), np.nan)

for i, pi in enumerate(providers_order):
    for j, pj in enumerate(providers_order):
        if i == j:
            agree_matrix[i, j] = 100.0
            continue
        # dropna sur les deux colonnes séparément pour éviter les doublons
        col_i = pivot_agree[pi].dropna()
        col_j = pivot_agree[pj].dropna()
        common = col_i.index.intersection(col_j.index)
        if len(common) == 0:
            continue
        vi = col_i.loc[common].values.astype(int)
        vj = col_j.loc[common].values.astype(int)
        agree_matrix[i, j] = float((vi == vj).mean() * 100)

agree_df = pd.DataFrame(agree_matrix, index=providers_order, columns=providers_order)
labels = [SHORT[p] for p in providers_order]

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(
    agree_df,
    annot=True, fmt=".0f", annot_kws={"size": 8},
    cmap="YlOrRd",
    vmin=0, vmax=100,
    xticklabels=labels, yticklabels=labels,
    linewidths=0.4, linecolor="white",
    ax=ax,
    cbar_kws={"label": "Accord (%)"},
)
ax.set_title("Accord pair-à-pair entre modèles (% de prompts avec le même choix)", fontsize=12, fontweight="bold")
plt.xticks(rotation=30, ha="right", fontsize=8)
plt.yticks(rotation=0, fontsize=8)
plt.tight_layout()
plt.savefig("results/g3_pairwise_agreement.png", dpi=150)
plt.show()


### Graphique 4 — Carte de chaleur des réponses par prompt (288 prompts × modèles)

In [ ]:
MIN_COVERAGE_G4 = 0.80  # ignorer providers avec < 80 % de réponses valides

n_prompts = df["user_prompt"].nunique()
coverage = df.groupby("provider")["user_prompt"].nunique() / n_prompts
providers_g4 = [p for p in providers_order if coverage.get(p, 0) >= MIN_COVERAGE_G4]
excluded_g4 = [p for p in providers_order if p not in providers_g4]
if excluded_g4:
    print(f"Providers exclus (< {MIN_COVERAGE_G4*100:.0f}% de couverture) : {[SHORT[p] for p in excluded_g4]}")

pivot_resp = (
    df.groupby(["user_prompt", "provider"])["chosen_index"]
      .agg(lambda x: x.mode()[0] if len(x) > 0 else np.nan)
      .unstack("provider")
      .reindex(columns=providers_g4)
)

# Trier les prompts par indice majoritaire médian pour regrouper les patterns
pivot_resp = pivot_resp.loc[pivot_resp.median(axis=1).sort_values().index]

fig, ax = plt.subplots(figsize=(12, 14))
cmap = plt.cm.get_cmap("RdYlGn", 5)
bounds = [0.5, 1.5, 2.5, 3.5, 4.5, 5.5]
norm = mcolors.BoundaryNorm(bounds, cmap.N)

im = ax.imshow(pivot_resp.values, aspect="auto", cmap=cmap, norm=norm, interpolation="nearest")
ax.set_xticks(range(len(providers_g4)))
ax.set_xticklabels([SHORT[p] for p in providers_g4], rotation=30, ha="right", fontsize=9)
ax.set_yticks([])
ax.set_ylabel(f"{len(pivot_resp)} prompts (triés par choix médian)", fontsize=10)
ax.set_title("Choix par prompt et par modèle (couleur = indice 1–5)", fontsize=12, fontweight="bold")

cbar = fig.colorbar(im, ax=ax, ticks=[1, 2, 3, 4, 5], pad=0.02)
cbar.ax.set_yticklabels(["1", "2", "3", "4", "5"])
cbar.set_label("Itinéraire choisi")

plt.tight_layout()
plt.savefig("results/g4_heatmap_prompts.png", dpi=150)
plt.show()


### Graphique 6 — Analyse des Correspondances Multiples (ACM) en 2D

In [ ]:
try:
    from prince import MCA
    _has_prince = True
except ImportError:
    _has_prince = False
    print("Package 'prince' non installé — install via: pip install prince")
    print("Fallback : PCA sur les choix numériques")

# Matrice prompts × modèles (valeur = mode majoritaire du provider sur ce prompt)
X_raw = (
    df.groupby(["user_prompt", "provider"])["chosen_index"]
      .agg(lambda x: int(x.mode().iloc[0]))
      .unstack("provider")
      .reindex(columns=providers_order)
)

# Remplir les NaN avec le choix le plus fréquent du provider (pas de dropna total)
X = X_raw.apply(lambda col: col.fillna(col.mode().iloc[0] if col.notna().any() else 3)).astype(int)
print(f"Matrice ACM/PCA : {X.shape[0]} prompts × {X.shape[1]} modèles")

if X.empty or X.shape[0] < 2:
    print("Pas assez de données pour l'ACM/PCA.")
elif _has_prince:
    X_cat = X.astype(str)
    mca = MCA(n_components=2, random_state=42)
    coords = mca.fit_transform(X_cat.T)  # lignes = modèles
    coords.index = providers_order
    x_label, y_label = "Composante 1 (ACM)", "Composante 2 (ACM)"
    _plot_ok = True
else:
    from sklearn.decomposition import PCA
    from sklearn.preprocessing import StandardScaler
    Xmat = X.T.values.astype(float)  # shape (n_providers, n_prompts)
    Xn = StandardScaler().fit_transform(Xmat)
    n_comp = min(2, Xn.shape[0] - 1, Xn.shape[1])
    pca = PCA(n_components=n_comp, random_state=42)
    Xp = pca.fit_transform(Xn)
    if Xp.shape[1] < 2:
        Xp = np.hstack([Xp, np.zeros((Xp.shape[0], 2 - Xp.shape[1]))])
    coords = pd.DataFrame(Xp, index=providers_order, columns=[0, 1])
    x_label = f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)"
    y_label = f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)" if n_comp > 1 else "PC2"
    _plot_ok = True

if _plot_ok:
    fig, ax = plt.subplots(figsize=(9, 7))
    for p in providers_order:
        x, y = float(coords.loc[p, 0]), float(coords.loc[p, 1])
        ax.scatter(x, y, s=90, zorder=3)
        ax.annotate(SHORT[p], (x, y), textcoords="offset points", xytext=(6, 4), fontsize=8)

    ax.axhline(0, color="gray", linewidth=0.5, linestyle="--")
    ax.axvline(0, color="gray", linewidth=0.5, linestyle="--")
    ax.set_xlabel(x_label)
    ax.set_ylabel(y_label)
    ax.set_title("Position des modèles dans l'espace des choix (ACM / PCA)", fontsize=12, fontweight="bold")
    ax.spines[["top", "right"]].set_visible(False)
    plt.tight_layout()
    plt.savefig("results/g6_mca_scatter.png", dpi=150)
    plt.show()


### Graphique 7 — Verbosité vs Taux de divergence par rapport à la majorité

In [ ]:
# Verbosité : longueur moyenne de la justification par modèle
verbosity = df.groupby("provider")["reason"].apply(lambda s: s.str.len().mean()).reindex(providers_order)

# Vote majoritaire par prompt (sur chosen_index)
majority = (
    df.groupby(["user_prompt", "provider"])["chosen_index"]
      .agg(lambda x: x.mode()[0])
      .unstack("provider")
      .reindex(columns=providers_order)
)
prompt_majority = majority.mode(axis=1)[0]  # vote majoritaire par prompt

# Taux de divergence : % de prompts où le modèle diverge du vote majoritaire
divergence = {}
for p in providers_order:
    col = majority[p].dropna()
    maj = prompt_majority.reindex(col.index)
    divergence[p] = (col != maj).mean() * 100
divergence = pd.Series(divergence).reindex(providers_order)

fig, ax = plt.subplots(figsize=(9, 6))
for p in providers_order:
    ax.scatter(verbosity[p], divergence[p], s=80, zorder=3)
    ax.annotate(SHORT[p], (verbosity[p], divergence[p]),
                textcoords="offset points", xytext=(6, 4), fontsize=8)

# Ligne de tendance
m_mask = ~(np.isnan(verbosity) | np.isnan(divergence))
if m_mask.sum() > 1:
    z = np.polyfit(verbosity[m_mask], divergence[m_mask], 1)
    xline = np.linspace(verbosity[m_mask].min(), verbosity[m_mask].max(), 100)
    ax.plot(xline, np.poly1d(z)(xline), "r--", linewidth=1, alpha=0.6, label="Tendance")

ax.set_xlabel("Longueur moyenne de la justification (caractères)")
ax.set_ylabel("Taux de divergence vs majorité (%)")
ax.set_title("Verbosité vs Divergence par rapport au vote majoritaire", fontsize=12, fontweight="bold")
ax.legend(fontsize=9)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.savefig("results/g7_verbosity_divergence.png", dpi=150)
plt.show()

print("\nRésumé verbosité / divergence :")
summary = pd.DataFrame({"verbosity_mean": verbosity, "divergence_pct": divergence})
print(summary.to_string())


In [ ]:
OUTPUT_CSV = Path("results") / "influence_results_persona_v2.csv"
df_raw = pd.read_csv(OUTPUT_CSV)
df_raw
df_car = df_raw[df_raw["mode"].isin(["voiture", "car", "conducteur"])]
print(f"Prompts avec choix majoritaire 'Voiture' : {df_car['user_prompt'].nunique()} prompts uniques")
df_car.to_csv("results/prompts_choix_voiture.csv", index=False)